# Godot Coder From Scratch — full training
This notebook creates a new tokenizer and a randomly initialized GPT-style model. Select a GPU before running.

In [ ]:
!nvidia-smi
!git clone https://github.com/genity5-collab/godot-coder-from-scratch.git
%cd godot-coder-from-scratch
# Keep the platform-provided PyTorch/CUDA build; this is text-only training.
!pip uninstall -y -q torchaudio torchvision || true
!pip install -q -U transformers datasets tokenizers accelerate huggingface_hub safetensors

## 1. Download official training data
The docs are downloaded at runtime from the official Godot documentation repository. The downloader records provenance and license metadata.

In [ ]:
!python src/download_official_godot_docs.py --output data/raw/godot-docs --ref master
!python src/prepare_data.py --input data/raw/godot-docs --output data/processed
!wc -c data/processed/*.txt

## 2. Train a tokenizer from scratch

In [ ]:
!python src/train_tokenizer.py --input data/processed/train.txt --output tokenizer/godot-bpe --vocab-size 12000

## 3. Smoke test
Run this first. It verifies tokenization, packing, random initialization, checkpointing, and generation.

In [ ]:
!python src/train_model.py --data data/processed --tokenizer tokenizer/godot-bpe/tokenizer.json --output checkpoints/smoke-test --context-length 512 --epochs 1 --max-steps 50 --batch-size 1 --grad-accumulation 8 --save-steps 25 --eval-steps 25

In [ ]:
!ls -la checkpoints/smoke-test
!python evaluation/generate.py --model checkpoints/smoke-test --prompt 'Write a Godot 4 Node2D script that prints Hello Godot.'

## 4. Full training
This is the real run. Save the checkpoints directory to persistent storage before the runtime ends.

In [ ]:
!python src/train_model.py --data data/processed --tokenizer tokenizer/godot-bpe/tokenizer.json --output checkpoints/godot-coder-30m --context-length 1024 --epochs 3 --batch-size 2 --grad-accumulation 16 --learning-rate 0.0003 --save-steps 250 --eval-steps 250

## 5. Resume after disconnect
Restore the `checkpoints/godot-coder-30m` directory first, then run this cell. The script selects the latest checkpoint automatically.

In [ ]:
!python src/train_model.py --data data/processed --tokenizer tokenizer/godot-bpe/tokenizer.json --output checkpoints/godot-coder-30m --resume

## 6. Upload the finished model
Create an `HF_TOKEN` secret in Colab or Kaggle. Never paste a token into this notebook or commit it.

In [ ]:
# Colab: from google.colab import userdata; import os; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# Kaggle: from kaggle_secrets import UserSecretsClient; import os; os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
# !test -f checkpoints/godot-coder-30m/config.json
# !python src/upload_hf.py --model-dir checkpoints/godot-coder-30m --repo-id Dosmallelfso/godot-coder-30m --private